# DSA 504 — Class 15
## Shell Scripting I: Bash Fundamentals

**Date:** Monday, Oct 26

---

> **Scheduling note:** this session now sits in the slot originally held by the practical midterm, which has moved to Wednesday, Oct 28. This notebook covers what was originally planned for Class 16 (Shell Scripting I). If this swap isn't what you intended, let me know and I'll adjust.

Every class so far has used Python (and SQL) running inside a notebook. Today introduces a different, equally important tool: the **command line** — typing commands directly to your computer, instead of clicking through folders and menus. This is how real data work often automates repetitive tasks that don't need all of Python's machinery.

**A note on how this notebook works:** cells in this notebook starting with `%%bash` run real shell commands, not Python — this works because Jupyter can hand a cell off to bash to execute. Everything you see demonstrated here, you can also type directly into your own Terminal (Mac/Linux) or Git Bash/WSL (Windows) outside of Jupyter entirely.

### Learning goals
By the end of this class, you will be able to:
- Navigate the file system from the command line (`pwd`, `ls`, `cd`)
- Create, copy, move, and remove files and directories
- View file contents from the command line (`cat`, `head`, `tail`)
- Redirect command output into a file, and chain commands together with pipes
- Check and understand basic file permissions


## 1. Navigating the File System

Three commands you'll use constantly:
- `pwd` — **p**rint **w**orking **d**irectory (where am I right now?)
- `ls` — **l**i**s**t the contents of the current directory
- `cd` — **c**hange **d**irectory (move somewhere else)


In [ ]:
%%bash
pwd

In [ ]:
%%bash
ls

### Useful `ls` options


In [ ]:
%%bash
ls -l    # long format: permissions, size, modified date

In [ ]:
%%bash
ls -la   # also show hidden files (names starting with a dot)

### Moving around with `cd`


In [ ]:
%%bash
mkdir -p ~/dsa504_demo
cd ~/dsa504_demo
pwd

**A few shortcuts worth knowing:**
- `cd ..` — move up one directory (to the parent folder)
- `cd ~` — jump straight to your home directory
- `cd -` — jump back to the previous directory you were in
- `cd` with no arguments also goes to your home directory


In [ ]:
%%bash
cd ~/dsa504_demo
mkdir -p subfolder
cd subfolder
pwd
cd ..
pwd

## 2. Creating, Copying, Moving, and Removing

- `mkdir` — make a new directory
- `touch` — create a new, empty file (or update an existing file's timestamp)
- `cp` — copy a file
- `mv` — move (or rename) a file
- `rm` — remove (delete) a file — **there is no undo, no recycle bin**


In [ ]:
%%bash
cd ~/dsa504_demo
touch notes.txt
ls -l

In [ ]:
%%bash
cd ~/dsa504_demo
cp notes.txt notes_backup.txt
ls

In [ ]:
%%bash
cd ~/dsa504_demo
mv notes_backup.txt archive.txt
ls

**`mv` is also how you rename a file** — there's no separate "rename" command; moving a file to a new name in the same folder is exactly what renaming means to the shell.


In [ ]:
%%bash
cd ~/dsa504_demo
rm archive.txt
ls

**Removing a directory** requires a different command — `rmdir` for an empty one, or `rm -r` (recursive) for one with contents. **Be careful with `rm -r`** — it deletes everything inside without asking, and there's no confirmation prompt or trash bin to recover from, unlike deleting a file in Finder or File Explorer.


In [ ]:
%%bash
cd ~/dsa504_demo
mkdir empty_folder
rmdir empty_folder
ls

## 3. Viewing File Contents

First, let's create a small file with some real content to look at.


In [ ]:
%%bash
cd ~/dsa504_demo
printf "Utica,4521.75\nAlbany,3980.10\nRome,5102.00\nSyracuse,3340.00\n" > stores.csv
cat stores.csv

`cat` ("concatenate") prints the whole file. For a large file, that's not very useful — `head` and `tail` show just the beginning or end.


In [ ]:
%%bash
cd ~/dsa504_demo
head -n 2 stores.csv

In [ ]:
%%bash
cd ~/dsa504_demo
tail -n 2 stores.csv

`head`/`tail` are especially useful on large files — opening a multi-gigabyte log file entirely with `cat` could freeze your terminal, while `head` only reads the first few lines.


## 4. Redirection — Sending Output to a File

By default, a command's output prints to your screen. **Redirection** sends it somewhere else instead — usually into a file.

- `>` — write output to a file, **overwriting** it if it already exists
- `>>` — **append** output to the end of a file, keeping what's already there


In [ ]:
%%bash
cd ~/dsa504_demo
echo "This is a new file" > output.txt
cat output.txt

In [ ]:
%%bash
cd ~/dsa504_demo
echo "This line overwrites everything before it" > output.txt
cat output.txt

In [ ]:
%%bash
cd ~/dsa504_demo
echo "This line gets appended" >> output.txt
echo "So does this one" >> output.txt
cat output.txt

**The difference between `>` and `>>` is one of the most common shell mistakes** — using `>` when you meant `>>` silently destroys whatever was in the file before, with no warning and no undo.


## 5. Pipes — Chaining Commands Together

A **pipe** (`|`) sends the *output* of one command directly into the *input* of the next command — this is how small, simple commands combine into something more powerful, without needing to save results to a file in between.


In [ ]:
%%bash
cd ~/dsa504_demo
cat stores.csv | wc -l

`wc -l` counts lines (**w**ord **c**ount, `-l` for lines). Piped together, this reads as: "take the contents of `stores.csv`, and count how many lines it has" — one command's output became the next command's input, with no temporary file needed.


In [ ]:
%%bash
cd ~/dsa504_demo
cat stores.csv | sort

In [ ]:
%%bash
cd ~/dsa504_demo
cat stores.csv | sort -t',' -k2 -n

That last command sorts `stores.csv` numerically (`-n`) by the second column (`-k2`), treating commas as the separator (`-t','`) — notice this is conceptually the exact same idea as `sort_values()` in pandas or `ORDER BY` in SQL, just expressed in a third syntax. The *concept* (sort by a specific field) is one you already know; only the specific command syntax is new.


In [ ]:
%%bash
cd ~/dsa504_demo
cat stores.csv | sort -t',' -k2 -n | head -n 2

**Chaining three commands together**: sort the file, then take just the top 2 lines — "lowest 2 stores by the second column." Pipes can chain as many commands together as you need, each one feeding the next.


## 6. File Permissions (a First Look)

Every file has permissions controlling who can read, write, or execute it. You saw these earlier in the `ls -l` output — Class 16 (text processing) and automation work later this semester will use these more directly, but it's worth recognizing them now.


In [ ]:
%%bash
cd ~/dsa504_demo
ls -l stores.csv

**Reading the permission string** (e.g., `-rw-r--r--`): the first character indicates file type (`-` for a regular file, `d` for a directory). The next nine characters are three groups of three — **r**ead, **w**rite, e**x**ecute — for the file's owner, its group, and everyone else, in that order. `rw-r--r--` means the owner can read and write, while the group and everyone else can only read.


In [ ]:
%%bash
cd ~/dsa504_demo
chmod +x stores.csv
ls -l stores.csv

In [ ]:
%%bash
cd ~/dsa504_demo
chmod -x stores.csv
ls -l stores.csv

`chmod` ("change mode") changes permissions — `+x` adds execute permission, `-x` removes it. You'll use this for real in a few classes, when a script needs execute permission to be run directly rather than passed to an interpreter.


---
## Guided Practice

Work through these in your own Terminal (or continue using `%%bash` cells here). All of these build on the `~/dsa504_demo` folder from above.


### Exercise 1 — Navigation and creation
Create a new directory called `practice` inside `~/dsa504_demo`, move into it, and create two empty files inside it called `a.txt` and `b.txt`. Confirm with `ls`.


In [ ]:
%%bash
# Exercise 1 — your commands here



### Exercise 2 — Redirection
Inside `practice`, write the text `"first line"` into a file called `log.txt` using `>`. Then append `"second line"` to the same file using `>>`. Print the final contents with `cat`.


In [ ]:
%%bash
# Exercise 2 — your commands here



### Exercise 3 — Pipes
Using `stores.csv` from earlier (one directory up, in `~/dsa504_demo`), write a single piped command that sorts the file alphabetically by the first column and shows only the first 2 results.


In [ ]:
%%bash
# Exercise 3 — your commands here



### Exercise 4 (stretch) — Combine everything
Starting from `~/dsa504_demo/practice`: create a new file called `summary.txt`, write a short line into it, check its permissions with `ls -l`, then remove the `a.txt` and `b.txt` files you no longer need, leaving only `log.txt` and `summary.txt`.


In [ ]:
%%bash
# Exercise 4 — your commands here



---
## Solutions


---
## Wrap-up

**Recap:** navigating with `pwd`/`ls`/`cd`; creating and managing files and directories (`mkdir`, `touch`, `cp`, `mv`, `rm`); viewing file contents (`cat`, `head`, `tail`); redirecting output to a file (`>` overwrites, `>>` appends); chaining commands with pipes (`|`); and a first look at file permissions with `ls -l` and `chmod`.

**Common mistakes to watch for:**
- Using `>` when you meant `>>`, silently overwriting a file's previous contents
- Running `rm` (especially `rm -r`) without double-checking you're in the right directory first — there is no undo
- Forgetting that `cd` with no path takes you home, which can be surprising if you expected to stay put
- Confusing `rmdir` (only works on empty directories) with `rm -r` (works on anything, including non-empty folders)

**Before next class (Nov 2):** Class 16 (originally numbered Class 17) covers shell text-processing tools — `grep`, `sed`, `awk`, `cut` — for searching and transforming data directly from the command line. **HW4 is due before this class.**
